# 05.03 CatBoost Classifier Benchmark (Stratified 10-Fold CV)

**Phase 05.03 — Gradient Boosting Models (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Train and benchmark an optimized CatBoost model on the 48 tabular features using Stratified 10-Fold Cross-Validation loaded from `data/processed/splits/cv_split_indices.json`. Generates comprehensive publication-ready evaluation charts (fold trajectories, metric stability error bars, confusion matrix, ROC curve, per-class performance, and gain-based feature importances).

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Establish regularized CatBoost performance across exactly 10 Stratified folds ($K=10$).
2. Report fold mean and standard deviation for: Accuracy, Macro Precision, Macro Recall, Macro F1, Weighted F1, Per-Class Precision, and Per-Class Recall.
3. Compute out-of-fold confusion matrix, normalized confusion matrix, and ROC-AUC curve.
4. Save metrics to `models/trained/gradient_boosting/02_catboost_metrics.json` and export high-resolution (300 DPI PNG & PDF) visualization figures to `reports/phase_05/visualizations/02_catboost/`.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary tabular features)
- `data/processed/splits/cv_split_indices.json` (Stratified 10-Fold split definitions)

**Expected Outputs:**
- `models/trained/gradient_boosting/02_catboost_metrics.json`
- `reports/phase_05/visualizations/02_catboost/10fold_performance_metrics.{png,pdf}`
- `reports/phase_05/visualizations/02_catboost/confusion_matrix_and_roc.{png,pdf}`
- `reports/phase_05/visualizations/02_catboost/per_class_performance.{png,pdf}`
- `reports/phase_05/visualizations/02_catboost/feature_importance.{png,pdf}`
- `reports/phase_05/visualizations/02_catboost/evaluation_summary_table.csv`

### 1. Environment Setup & Library Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
import catboost as cb
from catboost import CatBoostClassifier
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]
print(f"Libraries loaded successfully. RANDOM_SEED={RANDOM_SEED}")


### 2. Setup Paths & Load Data and 10-Fold Indices

In [ ]:
PROJECT_ROOT = Path("../../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
CV_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "cv_split_indices.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "trained" / "gradient_boosting"
REPORT_VIZ_DIR = PROJECT_ROOT / "reports" / "phase_05" / "visualizations" / "02_catboost"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_VIZ_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert CV_PATH.exists(), f"Missing CV split indices: {CV_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(CV_PATH, "r", encoding="utf-8") as f:
    cv_data = json.load(f)

feature_cols = cv_data["feature_columns"]
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_full = df_train[feature_cols]
y_train_full = (df_train["eligibility"] == "ELIGIBLE").astype(int)

assert len(cv_data["folds"]) == 10, f"Expected 10 folds, found {len(cv_data['folds'])}"
print(f"Loaded training development set: X={X_train_full.shape}, y={y_train_full.shape}")
print(f"Stratified 10-Fold CV indices confirmed: {len(cv_data['folds'])} folds")


### 3. Hyperparameter Tuning Across Stratified 10-Fold CV

In [ ]:
param_candidates = [
    {"iterations": 150, "depth": 4, "learning_rate": 0.05},
    {"iterations": 150, "depth": 6, "learning_rate": 0.05},
    {"iterations": 300, "depth": 4, "learning_rate": 0.03},
    {"iterations": 300, "depth": 6, "learning_rate": 0.03}
]
grid_results = []
for params in param_candidates:
    fold_f1s = []
    for fold_info in cv_data["folds"]:
        t_idx, v_idx = fold_info["train_indices"], fold_info["val_indices"]
        clf = CatBoostClassifier(iterations=params["iterations"], depth=params["depth"], learning_rate=params["learning_rate"], loss_function="Logloss", verbose=0, random_seed=RANDOM_SEED)
        clf.fit(X_train_full.iloc[t_idx], y_train_full.iloc[t_idx])
        fold_f1s.append(f1_score(y_train_full.iloc[v_idx], clf.predict(X_train_full.iloc[v_idx]), average="macro", zero_division=0))
    grid_results.append({"params": params, "mean_macro_f1": float(np.mean(fold_f1s))})
best_cfg = max(grid_results, key=lambda x: x["mean_macro_f1"])
best_params = best_cfg["params"]
print(f"Optimal CatBoost params selected: {best_params} with Mean Macro F1: {best_cfg['mean_macro_f1']:.4f}")


### 4. Full Stratified 10-Fold Cross-Validation Evaluation

In [ ]:
fold_evaluations = []
oof_predictions = np.zeros(len(y_train_full))
oof_probabilities = np.zeros(len(y_train_full))

start_time = time.time()
for fold_info in cv_data["folds"]:
    f_num = fold_info["fold"]
    t_idx = fold_info["train_indices"]
    v_idx = fold_info["val_indices"]
    
    X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
    X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
    
    model_pipeline = CatBoostClassifier(iterations=best_params['iterations'], depth=best_params['depth'], learning_rate=best_params['learning_rate'], loss_function='Logloss', verbose=0, random_seed=RANDOM_SEED)
    model_pipeline.fit(X_tr, y_tr)
    val_preds = model_pipeline.predict(X_va)
    val_probs = model_pipeline.predict_proba(X_va)[:, 1]
    
    oof_predictions[v_idx] = val_preds
    oof_probabilities[v_idx] = val_probs
    
    f_acc = accuracy_score(y_va, val_preds)
    f_prec_macro = precision_score(y_va, val_preds, average="macro", zero_division=0)
    f_rec_macro = recall_score(y_va, val_preds, average="macro", zero_division=0)
    f_f1_macro = f1_score(y_va, val_preds, average="macro", zero_division=0)
    f_f1_weighted = f1_score(y_va, val_preds, average="weighted", zero_division=0)
    f_prec_c0 = precision_score(y_va, val_preds, pos_label=0, zero_division=0)
    f_rec_c0 = recall_score(y_va, val_preds, pos_label=0, zero_division=0)
    f_prec_c1 = precision_score(y_va, val_preds, pos_label=1, zero_division=0)
    f_rec_c1 = recall_score(y_va, val_preds, pos_label=1, zero_division=0)
    f_auc = roc_auc_score(y_va, val_probs)
    
    fold_evaluations.append({
        "fold": f_num,
        "accuracy": float(f_acc),
        "macro_precision": float(f_prec_macro),
        "macro_recall": float(f_rec_macro),
        "macro_f1": float(f_f1_macro),
        "weighted_f1": float(f_f1_weighted),
        "prec_ineligible": float(f_prec_c0),
        "rec_ineligible": float(f_rec_c0),
        "prec_eligible": float(f_prec_c1),
        "rec_eligible": float(f_rec_c1),
        "roc_auc": float(f_auc)
    })
    print(f"Fold {f_num:02d} -> Acc: {f_acc:.4f}, Macro-F1: {f_f1_macro:.4f}, AUC: {f_auc:.4f}, Rec-Elig: {f_rec_c1:.4f}")

total_duration_ms = (time.time() - start_time) * 1000
df_folds_metric = pd.DataFrame(fold_evaluations)

# Compute summary statistics
metrics_mean = df_folds_metric.mean(numeric_only=True).to_dict()
metrics_std = df_folds_metric.std(numeric_only=True).to_dict()
print(f"\nStratified 10-Fold CV Complete ({total_duration_ms:.1f} ms).")
print(f"Mean Macro F1: {metrics_mean['macro_f1']:.4f} +/- {metrics_std['macro_f1']:.4f}")


### 5. Evaluation Summary Table & Artifact Export

In [ ]:
# Construct standard summary table
summary_table_data = [
    {"Metric": "Accuracy", "Mean": f"{metrics_mean['accuracy']:.4f}", "Standard Deviation": f"{metrics_std['accuracy']:.4f}"},
    {"Metric": "Macro Precision", "Mean": f"{metrics_mean['macro_precision']:.4f}", "Standard Deviation": f"{metrics_std['macro_precision']:.4f}"},
    {"Metric": "Macro Recall", "Mean": f"{metrics_mean['macro_recall']:.4f}", "Standard Deviation": f"{metrics_std['macro_recall']:.4f}"},
    {"Metric": "Macro F1-score", "Mean": f"{metrics_mean['macro_f1']:.4f}", "Standard Deviation": f"{metrics_std['macro_f1']:.4f}"},
    {"Metric": "Weighted F1-score", "Mean": f"{metrics_mean['weighted_f1']:.4f}", "Standard Deviation": f"{metrics_std['weighted_f1']:.4f}"},
    {"Metric": "ROC-AUC", "Mean": f"{metrics_mean['roc_auc']:.4f}", "Standard Deviation": f"{metrics_std['roc_auc']:.4f}"},
    {"Metric": "Per-Class Precision (INELIGIBLE)", "Mean": f"{metrics_mean['prec_ineligible']:.4f}", "Standard Deviation": f"{metrics_std['prec_ineligible']:.4f}"},
    {"Metric": "Per-Class Recall (INELIGIBLE)", "Mean": f"{metrics_mean['rec_ineligible']:.4f}", "Standard Deviation": f"{metrics_std['rec_ineligible']:.4f}"},
    {"Metric": "Per-Class Precision (ELIGIBLE)", "Mean": f"{metrics_mean['prec_eligible']:.4f}", "Standard Deviation": f"{metrics_std['prec_eligible']:.4f}"},
    {"Metric": "Per-Class Recall (ELIGIBLE)", "Mean": f"{metrics_mean['rec_eligible']:.4f}", "Standard Deviation": f"{metrics_std['rec_eligible']:.4f}"}
]
df_summary_table = pd.DataFrame(summary_table_data)
df_summary_table.to_csv(REPORT_VIZ_DIR / "evaluation_summary_table.csv", index=False)

# Fit full model on development set for feature importances
full_cb = CatBoostClassifier(iterations=best_params['iterations'], depth=best_params['depth'], learning_rate=best_params['learning_rate'], loss_function='Logloss', verbose=0, random_seed=RANDOM_SEED)
full_cb.fit(X_train_full, y_train_full)
df_importance = pd.DataFrame({'feature': feature_cols, 'importance': full_cb.get_feature_importance()}).sort_values(by='importance', ascending=False).reset_index(drop=True)
df_importance.to_csv(REPORT_VIZ_DIR / 'feature_importances.csv', index=False)

# Save JSON metrics artifact
metrics_payload = {
    "model_name": "CatBoost",
    "model_type": "gradient_boosting_ordered",
    "cv_protocol": "StratifiedKFold-10Fold",
    "best_hyperparameters": best_params,
    "cv_metrics_mean": metrics_mean,
    "cv_metrics_std": metrics_std,
    "fold_details": fold_evaluations,
    "total_cv_time_ms": round(total_duration_ms, 2),
    "top_5_features": df_importance.head(5).to_dict(orient="records") if 'df_importance' in locals() else []
}
with open(OUTPUT_DIR / "02_catboost_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print("Evaluation summary table:\n", df_summary_table.to_string(index=False))


### 6. Visualizations & Statistical Graphs

Generates:
- **Figure 1:** 10-Fold Performance Metrics (Accuracy, Precision, Recall, Macro F1 across folds and Mean +/- Std).
- **Figure 2:** Out-of-Fold Confusion Matrix & ROC Curve.
- **Figure 3:** Per-Class Performance Breakdown (Precision, Recall, F1 for INELIGIBLE vs ELIGIBLE).
- **Figure 4:** Gradient Boosting Feature Importance Analysis.

In [ ]:
# --- FIGURE 1: 10-Fold Metric Trajectory & Stability ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

folds_x = [f"F{i}" for i in range(1, 11)]
ax1.plot(folds_x, df_folds_metric["accuracy"], marker="o", linewidth=2, label="Accuracy", color="#1b9e77")
ax1.plot(folds_x, df_folds_metric["macro_f1"], marker="s", linewidth=2, label="Macro F1", color="#d95f02")
ax1.plot(folds_x, df_folds_metric["macro_recall"], marker="^", linewidth=2, label="Macro Recall", color="#7570b3")
ax1.set_title(f"{model_display_name} — Fold-by-Fold Performance (K=10)", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Cross-Validation Fold", fontsize=10, fontweight="bold")
ax1.set_ylabel("Metric Score (0.0 - 1.0)", fontsize=10, fontweight="bold")
ax1.set_ylim(0.5, 1.05)
ax1.legend(loc="lower right", frameon=True)

# Summary Bar with Std Error Bars
eval_names = ["Accuracy", "Macro Prec", "Macro Rec", "Macro F1", "Weighted F1", "ROC-AUC"]
eval_means = [metrics_mean["accuracy"], metrics_mean["macro_precision"], metrics_mean["macro_recall"], metrics_mean["macro_f1"], metrics_mean["weighted_f1"], metrics_mean["roc_auc"]]
eval_stds = [metrics_std["accuracy"], metrics_std["macro_precision"], metrics_std["macro_recall"], metrics_std["macro_f1"], metrics_std["weighted_f1"], metrics_std["roc_auc"]]

bars = ax2.bar(eval_names, eval_means, yerr=eval_stds, capsize=4, color="#386cb0", width=0.55, edgecolor="black", linewidth=1.1)
ax2.set_title(f"{model_display_name} — Mean 10-Fold Metrics (+/- 1 Std)", fontsize=12, fontweight="bold", pad=10)
ax2.set_ylabel("Score", fontsize=10, fontweight="bold")
ax2.set_ylim(0.0, 1.15)
for bar in bars:
    h = bar.get_height()
    ax2.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 7), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "10fold_performance_metrics.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "10fold_performance_metrics.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 1: 10fold_performance_metrics (PNG/PDF)")


In [ ]:
# --- FIGURE 2: Out-of-Fold Confusion Matrix & ROC Curve ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_train_full, oof_predictions)
cm_norm = confusion_matrix(y_train_full, oof_predictions, normalize="true")

annot_matrix = np.empty_like(cm, dtype=object)
for r in range(2):
    for c in range(2):
        annot_matrix[r, c] = f"{cm[r, c]}\n({cm_norm[r, c]*100:.1f}%)"

sns.heatmap(cm_norm, annot=annot_matrix, fmt="", cmap="Blues", cbar=False, ax=ax1,
            xticklabels=["INELIGIBLE", "ELIGIBLE"], yticklabels=["INELIGIBLE", "ELIGIBLE"], linewidths=1.2, linecolor="white")
ax1.set_title(f"{model_display_name} — Out-of-Fold Confusion Matrix", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Predicted Class", fontsize=10, fontweight="bold")
ax1.set_ylabel("Actual Class", fontsize=10, fontweight="bold")

# ROC Curve
fpr, tpr, _ = roc_curve(y_train_full, oof_probabilities)
ax2.plot(fpr, tpr, color="#e41a1c", lw=2.2, label=f"OOF ROC (AUC = {metrics_mean['roc_auc']:.3f})")
ax2.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Chance (AUC = 0.500)")
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel("False Positive Rate", fontsize=10, fontweight="bold")
ax2.set_ylabel("True Positive Rate", fontsize=10, fontweight="bold")
ax2.set_title(f"{model_display_name} — Out-of-Fold ROC Curve", fontsize=12, fontweight="bold", pad=10)
ax2.legend(loc="lower right", frameon=True)

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "confusion_matrix_and_roc.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "confusion_matrix_and_roc.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 2: confusion_matrix_and_roc (PNG/PDF)")


In [ ]:
# --- FIGURE 3: Per-Class Performance Comparison ---
fig, ax = plt.subplots(figsize=(8, 5))

classes = ["INELIGIBLE (0)", "ELIGIBLE (1)"]
x = np.arange(len(classes))
width = 0.25

prec_vals = [metrics_mean["prec_ineligible"], metrics_mean["prec_eligible"]]
rec_vals = [metrics_mean["rec_ineligible"], metrics_mean["rec_eligible"]]
f1_vals = [
    2 * (prec_vals[0] * rec_vals[0]) / (prec_vals[0] + rec_vals[0] + 1e-9),
    2 * (prec_vals[1] * rec_vals[1]) / (prec_vals[1] + rec_vals[1] + 1e-9)
]

b1 = ax.bar(x - width, prec_vals, width, label="Precision", color="#66c2a5", edgecolor="black")
b2 = ax.bar(x, rec_vals, width, label="Recall", color="#fc8d62", edgecolor="black")
b3 = ax.bar(x + width, f1_vals, width, label="F1-Score", color="#8da0cb", edgecolor="black")

ax.set_title(f"{model_display_name} — Per-Class Performance Breakdown", fontsize=12, fontweight="bold", pad=10)
ax.set_ylabel("Score", fontsize=10, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=10, fontweight="bold")
ax.set_ylim(0.0, 1.15)
ax.legend(loc="upper right", frameon=True)

for bar_group in [b1, b2, b3]:
    for b in bar_group:
        h = b.get_height()
        ax.annotate(f"{h:.2f}", xy=(b.get_x() + b.get_width() / 2, h),
                    xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "per_class_performance.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "per_class_performance.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 3: per_class_performance (PNG/PDF)")


In [ ]:
# --- FIGURE 4: Top Predictor Importances ---
fig, ax = plt.subplots(figsize=(10, 6))

if 'df_importance' in locals() and not df_importance.empty:
    top15 = df_importance.head(15).iloc[::-1]
    bars = ax.barh(top15["feature"], top15["importance"], color="#fc8d59", edgecolor="black", linewidth=1.1)
    ax.set_title(f"{model_display_name} — Top 15 Primary Feature Importances", fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Feature Importance Score", fontsize=10, fontweight="bold")
    for b in bars:
        w = b.get_width()
        ax.annotate(f"{w:.4f}", xy=(w, b.get_y() + b.get_height() / 2),
                    xytext=(4, 0), textcoords="offset points", ha="left", va="center", fontsize=8, fontweight="bold")
else:
    ax.text(0.5, 0.5, "Model feature importance visualization ready for manual execution.",
            ha="center", va="center", fontsize=11, bbox=dict(boxstyle="round", facecolor="#f7f7f9"))
    ax.set_axis_off()

plt.tight_layout()
fig.savefig(REPORT_VIZ_DIR / "feature_importance.png", dpi=300, bbox_inches="tight")
fig.savefig(REPORT_VIZ_DIR / "feature_importance.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)
print("Exported Figure 4: feature_importance (PNG/PDF)")


### 7. Interpretation Guide for Researcher

- **Cross-Validation Stability:** Review Figure 1 (left) for fold-to-fold variance in CatBoost. Look for consistency across all 10 folds.
- **Error Asymmetry:** Examine Figure 2 (Confusion Matrix). Assess whether False Negatives (eligible charity patients classified as ineligible) are properly minimized.
- **Class Parity:** Figure 3 compares Precision and Recall across classes. A balanced model should not bias heavily towards either class.
- **Key Predictors:** Figure 4 displays which non-linear features dominate the gradient boosted tree splits.

### Completion Checklist

- [ ] Stratified 10-Fold CV ($K=10$) executed strictly on training partition.
- [ ] Mean and standard deviation computed for Accuracy, Macro Precision, Recall, and F1.
- [ ] Out-of-fold confusion matrix and ROC curve generated.
- [ ] Per-class metrics reported.
- [ ] Feature importance / weights extracted.
- [ ] Figures exported to PNG (300 DPI) and PDF.
- [ ] Summary table exported to CSV.
- [ ] Zero test set snooping confirmed.